In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import shutil

# Load file
df = pd.read_csv('/content/drive/MyDrive/BiasProject/data.csv')
print("--- Sample Data ---")
print(df.head())

df = df.dropna()

# ==========================================
# STEP 2: DATA PREPROCESSING & CLEANING
# ==========================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

df['text'] = df['text'].apply(clean_text)

# Stratified Split to maintain exact balanced target ratios
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"\n✅ Data Prepared!")
print(f"Total rows in dataset: {len(df)}")
print(f"Training on: {len(train_texts)} sentences")
print(f"Testing on: {len(test_texts)} sentences")

# ==========================================
# STEP 3: OPTIMIZED FAST TOKENIZATION
# ==========================================
print("\n⏳ Initializing Tokenizer...")
# use_fast=True leverages tokenizers written in Rust for parallel processing
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)

# OPTIMIZATION: We remove padding=True here to let the Data Collator pad dynamically
print("Tokenizing training array...")
train_encodings = tokenizer(list(train_texts), truncation=True, max_length=128)
print("Tokenizing testing array...")
test_encodings = tokenizer(list(test_texts), truncation=True, max_length=128)

# Initialize Data Collator for dynamic padding per batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Tokenization Configuration Complete!")

# ==========================================
# STEP 4: TORCH DATASET CREATION
# ==========================================
class BiasDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BiasDataset(train_encodings, train_labels)
test_dataset = BiasDataset(test_encodings, test_labels)
print("✅ PyTorch Datasets ready for model input!")

# ==========================================
# STEP 5: INITIALIZE DISTILBERT BRAIN
# ==========================================
print("\n⏳ Loading Pre-trained DistilBERT Architecture (2 Labels)...")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Verify if CUDA / T4 GPU is actually running
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ Model moved to operational device: {device}")
if device.type != 'cuda':
    print("⚠️ WARNING: Colab is running on CPU. Please go to Runtime -> Change runtime type -> Select T4 GPU!")

# ==========================================
# STEP 6: CONFIGURE TRAINING PARAMETERS (ACCELERATED)
# ==========================================
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,  # OPTIMIZATION: Increased from 16 to 32 to better utilize GPU memory
    per_device_eval_batch_size=32,
    warmup_steps=300,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=50,                # Reduced logging frequency to save communication overhead
    report_to="none",
    fp16=True,                       # OPTIMIZATION: Activates 16-bit floating point acceleration (Crucial for speed)
    dataloader_num_workers=2         # Helps load data in parallel paths
)

# ==========================================
# STEP 7: RUN THE ACTIVE TRAINING LOOP
# ==========================================
print("\n🚀 Starting Training Loop...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator # OPTIMIZATION: Supplies the dynamic padding module
)

trainer.train()
print("🎉 Model Training Cycle Complete!")

# ==========================================
# STEP 8: PERMANENTLY SAVE MODEL TO DRIVE
# ==========================================
print("\n⏳ Archiving weights locally inside Colab workspace...")
model.save_pretrained("./bias_model_final")
tokenizer.save_pretrained("./bias_model_final")

shutil.make_archive("bias_model_final", 'zip', "./bias_model_final")
print("✅ Local zip file generated.")

print("⏳ Exporting final model weights zip to Google Drive...")
shutil.copy("bias_model_final.zip", "/content/drive/MyDrive/BiasProject/bias_model_final.zip")
print("🎯 Done! The fresh model weights are securely stored on your Google Drive.")

Mounted at /content/drive
--- Sample Data ---
                                                text  label
0  blackopal but the idea they want to do anythin...      1
1  mixed love for the duckshawks game tonight roo...      1
2  two fucking points away god dammit i fucking h...      1
3                          wow what does your dad do      1
4  rt hughgoesthere just met this idiot who prono...      1

✅ Data Prepared!
Total rows in dataset: 35086
Training on: 28068 sentences
Testing on: 7018 sentences

⏳ Initializing Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Model Training Cycle Complete!

⏳ Archiving weights locally inside Colab workspace...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Local zip file generated.
⏳ Exporting final model weights zip to Google Drive...
🎯 Done! The fresh model weights are securely stored on your Google Drive.
